# Solutions — Higher-Order Functions & JSON Parsing (Medium 18)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_bookings     = spark.table("samples.wanderbricks.bookings")

## Solution 1 — transform: Double Each Array Element

In [ ]:
df_prices = spark.createDataFrame([(1,[10,20,30]),(2,[5,15,25])], ["id","prices"])
result_1 = df_prices.withColumn("doubled_prices", F.transform("prices", lambda x: x * 2))

## Solution 2 — filter (higher-order): Keep Elements > 12

In [ ]:
df_prices = spark.createDataFrame([(1,[10,20,30]),(2,[5,15,25])], ["id","prices"])
result_2 = df_prices.withColumn("filtered_prices", F.filter("prices", lambda x: x > 12))

## Solution 3 — aggregate: Sum Array Without explode

In [ ]:
df_vals = spark.createDataFrame([(1,[1,2,3]),(2,[10,20])], ["id","vals"])
result_3 = df_vals.withColumn("array_sum", F.aggregate("vals", F.lit(0), lambda acc, x: acc + x))

## Solution 4 — exists and forall

In [ ]:
df_nums = spark.createDataFrame([(1,[1,3,5]),(2,[2,4,6]),(3,[1,2,3])], ["id","nums"])
result_4 = (
    df_nums
    .withColumn("any_odd",  F.exists("nums", lambda x: x % 2 != 0))
    .withColumn("all_even", F.forall("nums", lambda x: x % 2 == 0))
)

## Solution 5 — zip_with: Element-Wise Multiply

In [ ]:
df_ab = spark.createDataFrame([(1,[1,2,3],[4,5,6]),(2,[7,8],[9,10])], ["id","a","b"])
result_5 = df_ab.withColumn("products", F.zip_with("a","b", lambda x,y: x*y))

## Solution 6 — Array Utilities on Real Data

In [ ]:
result_6 = (
    df_transactions
    .join(spark.table("samples.bakehouse.sales_franchises"), on="franchiseID")
    .groupBy("franchiseID")
    .agg(F.collect_list("product").alias("all_products"))
    .withColumn("unique_products_sorted", F.sort_array(F.array_distinct("all_products")))
    .withColumn("product_count",   F.size("unique_products_sorted"))
    .withColumn("products_string", F.array_join("unique_products_sorted", ", "))
    .select("franchiseID","unique_products_sorted","product_count","products_string")
)

## Solution 7 — Parse JSON String to Struct

In [ ]:
schema = T.StructType([
    T.StructField("name", T.StringType()),
    T.StructField("age",  T.IntegerType()),
])
df_json = spark.createDataFrame(
    [('{"name":"Alice","age":30}',),('{"name":"Bob","age":25}',)], ["json_str"])
result_7 = (
    df_json
    .withColumn("parsed", F.from_json("json_str", schema))
    .withColumn("name", F.col("parsed.name"))
    .withColumn("age",  F.col("parsed.age"))
    .select("json_str","parsed","name","age")
)

## Solution 8 — Map Utilities: Create, Keys, Values

In [ ]:
result_8 = (
    df_transactions
    .withColumn("product_price_map",
        F.create_map(F.lit("product"), F.col("product"),
                     F.lit("price"),   F.col("totalPrice").cast("string")))
    .withColumn("map_keys",   F.map_keys("product_price_map"))
    .withColumn("map_values", F.map_values("product_price_map"))
    .select("transactionID","product_price_map","map_keys","map_values")
    .limit(100)
)